<a href="https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — CTR declines as position worsens

The paper observed that weighted CTR decreases as content moves farther from the top search positions. CTR was 0.423% for the Top 3, 0.339% for positions 4–10, 0.325% for positions 11–20, 0.163% for positions 21–50, and 0.050% for positions 50+. The label comes from observed Google Search Console clicks, impressions, and position data. This supports a directional relationship between search visibility and click capture, but it does not prove that moving a page to a higher position will automatically cause a specific CTR increase.

### Finding 2 — Positions 11–20 are an important optimization zone

The paper identifies positions 11–20 as “Striking Distance” because these pages already have search demand and may benefit from relevance improvements that move them onto page 1. The label comes from observed position, impressions, and click performance in the portfolio. The validation supports this as a useful decision-support finding, but it should not be treated as proof that optimizing every page in positions 11–20 will cause higher CTR.

In [1]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

In [2]:
# Read the HF_TOKEN from colab
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!" if HF_TOKEN else "Token not found")

#connect DuckDB to Hugging face
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

print("Connected successfully!")

Token loaded successfully!
Connected successfully!


In [3]:
DATASET = "hf://datasets/FlyRank/internship-warehouse" # the dataset path

In [4]:
feature_df = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_impressions > 0
""").df()

print("Rows:", len(feature_df))
print("Columns:", feature_df.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'client_has_gsc', 'client_has_ga4']


In [5]:
feature_df["ctr"] = (
    feature_df["gsc_clicks"] / feature_df["gsc_impressions"]
)

In [6]:
#creating features
features = [
    "gsc_impressions",
    "gsc_sum_position",
    "client_has_gsc",
    "client_has_ga4"
]

In [7]:
target = "ctr"   # the target value

In [8]:
print("Available dates:")
print(feature_df["report_date"].min())
print(feature_df["report_date"].max())

print("\nAvailable months:")
print(
    feature_df["report_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

Available dates:
2026-03-01 00:00:00
2026-03-31 00:00:00

Available months:
report_date
2026-03    3611061
Freq: M, Name: count, dtype: int64


In [9]:
months = con.sql("""
SELECT DISTINCT month
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
ORDER BY month
""").df()

months

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month
0,2025-01
1,2025-02
2,2025-03
3,2025-04
4,2025-05
5,2025-06
6,2025-07
7,2025-08
8,2025-09
9,2025-10


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

# My model under an honest split

I used a time-aware split, training on earlier months and evaluating on a later June test period to avoid using future data to predict the past.

The baseline achieved MAE = 0.007251, RMSE = 0.036932, and R² = -0.002539. The Random Forest achieved MAE = 0.007279, RMSE = 0.037182, and R² = -0.016179.

Under this time-aware test, the Random Forest did not improve on the baseline. Its errors were slightly higher and its R² was more negative, so the model should be treated as directional decision-support rather than evidence of stronger CTR prediction.

In [10]:
# Load the modeling periods

train_df = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-0[1-4]/*.parquet'
)
WHERE gsc_impressions > 0
""").df()

val_df = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet'
)
WHERE gsc_impressions > 0
""").df()

test_df = con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/*.parquet'
)
WHERE gsc_impressions > 0
""").df()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train rows: 12531047
Validation rows: 4373422
Test rows: 3878937


In [11]:
# Create target (CTR) and feature sets

for df in [train_df, val_df, test_df]:
    df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

features = [
    "gsc_impressions",
    "gsc_sum_position",
    "client_has_gsc",
    "client_has_ga4"
]

X_train = train_df[features]
y_train = train_df["ctr"]

X_val = val_df[features]
y_val = val_df["ctr"]

X_test = test_df[features]
y_test = test_df["ctr"]

print("Features:", features)
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Features: ['gsc_impressions', 'gsc_sum_position', 'client_has_gsc', 'client_has_ga4']
Training shape: (12531047, 4)
Validation shape: (4373422, 4)
Test shape: (3878937, 4)


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Baseline: predict the training-set mean CTR
baseline_prediction = y_train.mean()

baseline_pred = np.full(len(y_test), baseline_prediction)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("Time-aware baseline results")
print("MAE :", baseline_mae)
print("RMSE:", baseline_rmse)
print("R²  :", baseline_r2)

Time-aware baseline results
MAE : 0.007251329331887943
RMSE: 0.03693171596579568
R²  : -0.0025399316686165463


In [18]:
# Recreate the 500k training sample

train_sample = train_df.sample(
    n=500_000,
    random_state=42
)

X_train_sample = train_sample[features]
y_train_sample = train_sample["ctr"]

print("Training sample:", X_train_sample.shape)

Training sample: (500000, 4)


In [21]:
from sklearn.ensemble import RandomForestRegressor

rf_model_time = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model_time.fit(X_train_sample, y_train_sample)

print("Number of trees:", len(rf_model_time.estimators_))
print("Random Forest training completed")

Number of trees: 100
Random Forest training completed


In [22]:
# Evaluate Random Forest on the full June test set

rf_pred_time = rf_model_time.predict(X_test)

rf_mae_time = mean_absolute_error(y_test, rf_pred_time)
rf_rmse_time = np.sqrt(mean_squared_error(y_test, rf_pred_time))
rf_r2_time = r2_score(y_test, rf_pred_time)

print("Time-aware Random Forest results")
print("MAE :", rf_mae_time)
print("RMSE:", rf_rmse_time)
print("R²  :", rf_r2_time)

Time-aware Random Forest results
MAE : 0.007278638219650558
RMSE: 0.037182091071070844
R²  : -0.01617925966922229


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

# Leakage audit

The final feature set contains `gsc_impressions`, `gsc_sum_position`, `client_has_gsc`, and `client_has_ga4`. The target is CTR, calculated as clicks divided by impressions, and `gsc_clicks` was not included as a model feature.

The time-aware split also keeps future periods out of training: training uses January–April 2026, validation uses May 2026, and the final test uses June 2026. Based on these checks, I found no obvious target or future-window leakage in the final feature set.

In [23]:
# Leakage audit for final feature set

feature_cols = [
    "gsc_impressions",
    "gsc_sum_position",
    "client_has_gsc",
    "client_has_ga4"
]

print("Final features:")
for col in feature_cols:
    print("-", col)

print("\nTarget:")
print("ctr = gsc_clicks / gsc_impressions")

print("\nFuture-data check:")
print("Training period:", train_df["report_date"].min(), "to", train_df["report_date"].max())
print("Validation period:", val_df["report_date"].min(), "to", val_df["report_date"].max())
print("Test period:", test_df["report_date"].min(), "to", test_df["report_date"].max())

Final features:
- gsc_impressions
- gsc_sum_position
- client_has_gsc
- client_has_ga4

Target:
ctr = gsc_clicks / gsc_impressions

Future-data check:
Training period: 2026-01-01 00:00:00 to 2026-04-30 00:00:00
Validation period: 2026-05-01 00:00:00 to 2026-05-31 00:00:00
Test period: 2026-06-01 00:00:00 to 2026-06-30 00:00:00


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

#Claim rewrite

My initial claim was that the Random Forest could predict CTR better using search performance features.

After validation under an honest time-aware split, the measured results do not support that claim. The Random Forest performed slightly worse than the baseline on the June test period.

A safer conclusion is: **The analysis observed that the selected search-performance features provided limited predictive improvement for CTR under the tested time-aware split. The result is directional decision-support, not evidence that these features can reliably predict future CTR.**

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.